# Stage 11 prereq — landmark extraction for the 122-signer paper-split dataset

Walks the 6 paper-split zips (`eng_{train,val,test}_{lex,nonlex}.zip`) and writes a per-clip .npz layout under `/kaggle/working/landmark_cache_122/`.  No training here.

**MediaPipe is pinned** to match the version that produced the 38-signer cache.  A version mismatch silently changes coordinate normalisation; the sanity-check Cell 5 catches that within ±10% feature mean / std.

**Disk discipline** (Kaggle's 20 GB /kaggle/working/ limit + 70 GB /kaggle/temp/):
- Streams from the input zips one at a time via Python's `zipfile` — no full extraction.
- Per-clip output is fp16 .npz (~6 KB each) -> ~250 MB total cache.
- `_DONE` markers per zip survive session disconnects.

**Wall-clock**: ~3 h CPU.  GPU off (MediaPipe is CPU-bound).

## After this kernel commits

Save Version -> Save & Run All.  Then upload `/kaggle/working/landmark_cache_122/` as a new private Kaggle dataset (e.g. `wita-full-english-landmark-cache`).  The Stage 11 training notebook attaches that dataset.

## Cell 1 — Install + clone (MediaPipe pinned)

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk scipy --quiet
# PIN MediaPipe to the same version that produced the 38-signer cache.
# If the sanity check (Cell 5) fails the ±10% drift bound, bump this.
!pip install 'mediapipe==0.10.14' --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')
import mediapipe; print(f'mediapipe version: {mediapipe.__version__}')

## Cell 2 — Locate the 6 paper-split zips

In [ ]:
import os, glob, re
from pathlib import Path

# ============================================================================
# MANUAL OVERRIDE (use this if auto-detection below fails).
# Map each source (path can be a .zip OR a directory) to (split, subset).
# Leave as None to auto-detect.
# Example:
#   MANUAL_MAPPING = {
#       '/kaggle/input/wita-full-122/eng_train_lex': ('train','lex'),
#       ...
#   }
MANUAL_MAPPING = None
# ============================================================================

INPUT_ROOTS = sorted(glob.glob('/kaggle/input/*'))
print('Mounted inputs:')
for r in INPUT_ROOTS:
    print(f'  {r}')

# ----- Diagnostic A: all .zip files anywhere under /kaggle/input/ -----
print('\n--- ALL .zip files under /kaggle/input/ (any depth) ---')
all_zips = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
for p in all_zips:
    size_mb = os.path.getsize(p) / 1e6
    print(f'  {size_mb:>8.1f} MB  {p}')
print(f'Found {len(all_zips)} total .zip files.')

# ----- Diagnostic B: top-2-level directory tree under each input root -----
print('\n--- Top-2-level directories under each input root ---')
for root in INPUT_ROOTS:
    for d1 in sorted(os.listdir(root)):
        p1 = os.path.join(root, d1)
        if os.path.isdir(p1):
            n2 = 0
            for d2 in sorted(os.listdir(p1))[:8]:
                p2 = os.path.join(p1, d2)
                kind = 'dir' if os.path.isdir(p2) else 'file'
                print(f'  {kind}  {p2}')
                n2 += 1
            remaining = max(0, len(os.listdir(p1)) - n2)
            if remaining:
                print(f'         ... ({remaining} more entries)')
        else:
            print(f'  file {p1}')

# ----- Permissive classifier (split/subset) -----
SPLIT_TOK  = r'(train|valid?|test|tst|tr|va|te)'
SUBSET_TOK = r'(non[_-]?lex|nonlex|lex|freq[_-]?word|non[_-]?freq[_-]?word)'
PAT_SPLIT_FIRST  = re.compile(rf'{SPLIT_TOK}[_-]{SUBSET_TOK}',  re.I)
PAT_SUBSET_FIRST = re.compile(rf'{SUBSET_TOK}[_-]{SPLIT_TOK}', re.I)

def _classify(name: str):
    n = os.path.basename(name).lower().replace('.zip', '')
    for pat, order in [(PAT_SPLIT_FIRST, 'split_first'),
                       (PAT_SUBSET_FIRST, 'subset_first')]:
        m = pat.search(n)
        if not m: continue
        if order == 'split_first':
            split_raw, subset_raw = m.group(1), m.group(2)
        else:
            subset_raw, split_raw = m.group(1), m.group(2)
        split_map = {'train':'train','tr':'train',
                     'val':'val','valid':'val','va':'val',
                     'test':'test','tst':'test','te':'test'}
        split = split_map.get(split_raw, split_raw)
        if re.search(r'non', subset_raw):
            subset = 'nonlex'
        elif 'freq' in subset_raw and 'non' not in subset_raw:
            subset = 'lex'
        elif subset_raw == 'lex':
            subset = 'lex'
        else:
            subset = None
        if split in ('train','val','test') and subset in ('lex','nonlex'):
            return split, subset
    return None

# ----- Find candidate sources (zips OR directories matching the pattern) ----
candidates: list[tuple[str, str]] = []   # (path, "zip"|"dir")
for p in all_zips:
    candidates.append((p, 'zip'))

# Walk directories at depth 1, 2, and 3 under each input root.
for root in INPUT_ROOTS:
    for depth in (1, 2, 3):
        # Build a glob pattern with `depth` levels of '*'.
        pat = os.path.join(root, *(['*'] * depth))
        for p in glob.glob(pat):
            if os.path.isdir(p):
                candidates.append((p, 'dir'))

# Deduplicate (path, kind) while preserving order.
seen = set(); uniq_candidates: list[tuple[str, str]] = []
for p, k in candidates:
    key = (p, k)
    if key not in seen:
        seen.add(key); uniq_candidates.append((p, k))

# ----- Manual override path -----
if MANUAL_MAPPING is not None:
    classifications: dict[str, tuple[str, str]] = {}
    kind_of: dict[str, str] = {}
    for p, lbl in MANUAL_MAPPING.items():
        classifications[p] = lbl
        kind_of[p] = 'dir' if os.path.isdir(p) else ('zip' if p.endswith('.zip') else 'unknown')
else:
    classifications = {}
    kind_of = {}
    for p, k in uniq_candidates:
        c = _classify(p)
        if c is not None:
            # Prefer dir over zip when both exist for the same (split, subset).
            existing = next((q for q, lbl in classifications.items() if lbl == c), None)
            if existing is None:
                classifications[p] = c
                kind_of[p] = k
            else:
                # If we already have a zip for this (split, subset) and the new
                # candidate is a dir at a deeper path, prefer the dir.
                if kind_of[existing] == 'zip' and k == 'dir':
                    del classifications[existing]; del kind_of[existing]
                    classifications[p] = c
                    kind_of[p]         = k

print(f'\nClassified {len(classifications)} source(s) (zip or dir):')
for p in sorted(classifications.keys(), key=lambda q: classifications[q]):
    print(f'  [{kind_of[p]:<3s}]  {p}  ->  {classifications[p][0]}/{classifications[p][1]}')

if len(classifications) != 6:
    print('\n!!! Expected 6 sources (train/val/test × lex/nonlex), got '
          f'{len(classifications)}.\n'
          'Either:\n'
          '  (a) attach the dataset (no inputs above) -> right panel -> Data -> Add Data\n'
          '  (b) edit MANUAL_MAPPING at the top of THIS cell with explicit paths.\n')
else:
    print('\n✅ Found 6 sources. Cell 3 will set up resume markers.')

## Cell 3 — Output paths + resume markers

In [ ]:
OUT_ROOT = '/kaggle/working/landmark_cache_122'
os.makedirs(OUT_ROOT, exist_ok=True)
MARKER_DIR = os.path.join(OUT_ROOT, '_markers')
os.makedirs(MARKER_DIR, exist_ok=True)

PATH_TO_LABEL = dict(classifications)
PATH_TO_KIND  = dict(kind_of)
assert len(PATH_TO_LABEL) == 6, (
    f'Need 6 sources, got {len(PATH_TO_LABEL)}.  Re-run Cell 2 '
    'with MANUAL_MAPPING set if auto-detect missed.'
)
covered  = sorted(PATH_TO_LABEL.values())
expected = sorted([(s, ss) for s in ('train','val','test') for ss in ('lex','nonlex')])
assert covered == expected, (
    f'Coverage mismatch.\n  expected: {expected}\n  got:      {covered}'
)

def parse_split_subset(p): return PATH_TO_LABEL[p]
def parse_kind(p):         return PATH_TO_KIND[p]
def marker_path(p):
    split, subset = parse_split_subset(p)
    return os.path.join(MARKER_DIR, f'{split}_{subset}_DONE')

print('Final source -> (split, subset) mapping:')
for p, (s, ss) in sorted(PATH_TO_LABEL.items(), key=lambda kv: (kv[1][0], kv[1][1])):
    mk = marker_path(p)
    print(f'  [{parse_kind(p):<3s}]  {os.path.basename(p):<40s} -> {s}/{ss}  '
          f'(marker {"exists" if os.path.exists(mk) else "missing"})')

SPLIT_ORDER = {'val': 0, 'test': 1, 'train': 2}
source_paths = sorted(PATH_TO_LABEL.keys(),
                      key=lambda p: (SPLIT_ORDER[PATH_TO_LABEL[p][0]],
                                     PATH_TO_LABEL[p][1]))

## Cell 4 — Extract per-clip landmarks  (~30 min per zip on Kaggle CPU)

Streams each zip in turn, writing `<SIGNER>__<clip_id>.npz` files.  Resume-aware via `_DONE` markers.

In [ ]:
from wita_v2.datasets.landmark_cache_122 import (
    extract_zip_per_clip_landmarks,
    extract_dir_per_clip_landmarks,
)
from wita_v2.datasets.skeleton_cache  import LandmarkExtractor

extractor = LandmarkExtractor()
all_stats = {}
for p in source_paths:
    split, subset = parse_split_subset(p)
    kind          = parse_kind(p)
    mk            = marker_path(p)
    if os.path.exists(mk):
        print(f'[skip] {split}/{subset} already done')
        continue
    print(f'\n>>> extracting {split}/{subset}  [{kind}]  from {p}')
    if kind == 'zip':
        stats = extract_zip_per_clip_landmarks(
            zip_path=p, out_dir=OUT_ROOT, split=split, subset=subset,
            lang='english', max_frames=64, T_native=32,
            extractor=extractor, overwrite=False,
        )
    else:
        stats = extract_dir_per_clip_landmarks(
            dir_path=p, out_dir=OUT_ROOT, split=split, subset=subset,
            lang='english', max_frames=64, T_native=32,
            extractor=extractor, overwrite=False,
        )
    all_stats[f'{split}_{subset}'] = stats
    with open(mk, 'w') as f:
        import json; json.dump(stats, f, indent=2, default=str)
extractor.close()
print('\nAll sources processed.')

## Cell 5 — Sanity check: feature shape + value range

In [ ]:
import numpy as np
from pathlib import Path
import random

all_npz = list(Path(OUT_ROOT).rglob('*.npz'))
print(f'Total .npz files: {len(all_npz)}')
assert len(all_npz) > 0, 'No clips extracted'

random.seed(42)
sample = random.sample(all_npz, min(100, len(all_npz)))
feats = np.stack([np.load(p, allow_pickle=False)['feature'].astype(np.float32) for p in sample])
print(f'feature shape per clip : {feats.shape[1:]}')
print(f'feature dtype          : {feats.dtype}')
print(f'feature mean           : {feats.mean():.4f}')
print(f'feature std            : {feats.std():.4f}')
print(f'feature min/max        : {feats.min():.4f} / {feats.max():.4f}')
assert feats.shape[1:] == (32, 190), f'Bad shape: {feats.shape[1:]}'
assert np.all(np.isfinite(feats)), 'Non-finite values present'

# Per-split counts.
for split in ('train', 'val', 'test'):
    for subset in ('lex', 'nonlex'):
        n = len(list((Path(OUT_ROOT) / split / subset).glob('*.npz')))
        print(f'  {split}/{subset:<7s}: {n}')

## Cell 6 — Commit kernel + next step

1. **Save Version -> Save & Run All**.  The committed kernel's output dataset contains `landmark_cache_122/`.
2. After it commits, go to **Datasets -> New Dataset -> Notebook Output**, name it `wita-full-english-landmark-cache`.
3. Attach that dataset to the Stage 11 training notebook (next kernel).